# Cross-Validation

Runs cross-validation with optuna tuned HPs for all 21 **model x representation** combinations

In [32]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [3]:
with open("random_stratified_splits_reg.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=345  test=62  folds=5
Device: cpu


In [46]:
X_fp_RU        = np.load("Vectors/X_fp_RU.npy")
X_fp_pooled_RU    = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly      = np.load("Vectors/X_fp_poly.npy")
X_fp_pooled_poly = np.load("Vectors/X_fp_pooled_poly.npy")


In [47]:
graphs_PBSG = torch.load("Graphs/graphs_PBSG.pt")
graphs_SMILES = torch.load("Graphs/graphs_SMILES.pt")
graphs_SMILESplusglobal = torch.load("Graphs/graphs_SMILES+global.pt")

In [48]:
y=np.load("Vectors/y.npy")

In [49]:
import scripts.CV
# Define your reps
graph_reps = {
    "PBSG":         graphs_PBSG,
    "SMILES":       graphs_SMILES,
    "SMILES+global": graphs_SMILESplusglobal,
}

vector_reps = {
    "FP RU":          X_fp_RU,
    "FP+pooled RU":   X_fp_pooled_RU,
    "FP poly":        X_fp_poly,
    "FP+pooled poly": X_fp_pooled_poly,
}

In [51]:
import importlib
import scripts.CV

importlib.reload(scripts.CV)

<module 'scripts.CV' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML_clean/Regressor/scripts/CV.py'>

In [53]:
from scripts.CV import (run_all_gnn_cv_reg, run_all_classical_cv_reg,
                    print_cv_summary_reg, load_tuned_hp_reg)

hp = load_tuned_hp_reg("best_hp_reg.json")

graph_reps = {
    "PBSG":          graphs_PBSG,
    "SMILES":        graphs_SMILES,
    "SMILES+global": graphs_SMILESplusglobal
}

vector_reps = {
    "FP+pooled RU":   X_fp_pooled_RU,
    "FP RU":          X_fp_RU,
    "FP+pooled poly": X_fp_pooled_poly,
    "FP poly":        X_fp_poly,
}

# Inject y into graphs first
for graphs in graph_reps.values():
    for i, g in enumerate(graphs):
        if g.y is None:
            import torch
            g.y = torch.tensor([y[i]], dtype=torch.float32)

gnn_results = run_all_gnn_cv_reg(graph_reps, folds, df, device,
                                  y=y, hp_dict=hp)
cls_results = run_all_classical_cv_reg(vector_reps, y, folds, df,
                                        hp_dict=hp)

print_cv_summary_reg(gnn_results + cls_results)

# Save results
import json
all_results = gnn_results + cls_results
with open("cv_results_reg_tuned.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k != "subgroups"}
               for r in all_results], f, indent=2)

  Using tuned HPs for gcn_pbsg

  GNN REG CV: GCN (PBSG)  seed=10  in_ch=85  edge_dim=1  meta_dim=13
    Epoch  20 | train_loss=0.0757 val_loss=0.1302 val_mae=0.364 val_r2=0.761
    Epoch  40 | train_loss=0.0694 val_loss=0.0979 val_mae=0.317 val_r2=0.824
    Epoch  60 | train_loss=0.0636 val_loss=0.0830 val_mae=0.295 val_r2=0.857
    Epoch  80 | train_loss=0.0588 val_loss=0.0876 val_mae=0.307 val_r2=0.848
    Epoch 100 | train_loss=0.0474 val_loss=0.0915 val_mae=0.311 val_r2=0.840
    Epoch 120 | train_loss=0.0468 val_loss=0.0948 val_mae=0.311 val_r2=0.835
    Early stop at epoch 134  best_val_r2=0.8575
  Fold 1: MAE=18.43  RMSE=26.43  R²=0.8575  [27s]
    Epoch  20 | train_loss=0.0619 val_loss=0.1170 val_mae=0.345 val_r2=0.675
    Epoch  40 | train_loss=0.0614 val_loss=0.1099 val_mae=0.324 val_r2=0.695
    Epoch  60 | train_loss=0.0471 val_loss=0.1213 val_mae=0.335 val_r2=0.639
    Epoch  80 | train_loss=0.0447 val_loss=0.1110 val_mae=0.315 val_r2=0.697
    Early stop at epoch 81  bes

In [54]:
from scripts.CV import print_cv_summary_reg
print_cv_summary_reg(all_results)


  REG CV SUMMARY  (sorted by R2)
Name                          MAE             RMSE            R²              Time    
--------------------------------------------------------------------------------------
GINE (PBSG)                   17.59 ± 2.21    25.95 ± 3.34    0.8401 ± 0.0407 4.1m    
GATV2 (PBSG)                  18.49 ± 0.94    26.10 ± 2.39    0.8370 ± 0.0402 2.8m    
XGB (FP+pooled RU)            15.77 ± 2.35    27.20 ± 3.61    0.8273 ± 0.0331 2s      
GCN (PBSG)                    18.83 ± 1.79    26.93 ± 2.63    0.8263 ± 0.0452 2.1m    
RF (FP+pooled poly)           17.54 ± 2.91    27.63 ± 5.74    0.8173 ± 0.0671 1s      
XGB (FP+pooled poly)          17.23 ± 2.22    28.06 ± 2.60    0.8155 ± 0.0282 1s      
GATV2 (SMILES)                20.95 ± 2.44    28.91 ± 3.64    0.8019 ± 0.0456 30.9m   
GINE (SMILES)                 21.59 ± 2.58    29.58 ± 2.68    0.7927 ± 0.0408 4.0m    
GCN (SMILES)                  21.53 ± 1.88    29.77 ± 2.03    0.7898 ± 0.0418 4.9m    
RF (FP+po

In [55]:
import pandas as pd
import numpy as np

def reg_results_to_df(results):
    rows = []
    for r in results:
        rows.append({
            "name":     r["name"],
            "mae":      r["mean"]["mae"],  "mae_std":  r["std"]["mae"],
            "rmse":     r["mean"]["rmse"], "rmse_std": r["std"]["rmse"],
            "r2":       r["mean"]["r2"],   "r2_std":   r["std"]["r2"],
            "time_s":   r.get("time_s", 0),
        })
    return pd.DataFrame(rows)

df_reg_cv = reg_results_to_df(all_results)
df_reg_cv.to_csv("cv_results_reg.csv", index=False)
print("Saved → cv_results_reg.csv")

Saved → cv_results_reg.csv


In [56]:
import pandas as pd

def fold_results_to_df(results):
    rows = []
    for r in results:
        for fold_i, f in enumerate(r["folds"]):
            rows.append({
                "name":     r["name"],
                "fold":  fold_i + 1,
                "mae":      f["mae"],  
                "rmse":     f["rmse"],
                "r2":       f["r2"],   
            })
    return pd.DataFrame(rows)


df_folds = fold_results_to_df(all_results)
df_folds.to_csv("cv_results_folds.csv", index=False)